# Multi-interaction LLM system

In [1]:
import os
from dotenv import load_dotenv

In [2]:
model_name = os.getenv("MODEL_NAME")
model_key=os.getenv("API_KEY")

In [6]:

from openai import OpenAI
import dspy
kode_model = dspy.LM(
    f"openai/{model_name}", 
    api_key=model_key,
    base_url = "https://api.ai.kodekloud.com/v1"
)

dspy.configure(lm=kode_model )

In [ ]:
import mlflow
mlflow.dspy.autolog()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("DSPY EXPT")

In [10]:
class JokeSignature(dspy.Signature):
    """ Yor are a comedian who likes to tell stories before delivering a punchline. You should sound funny."""
    quest: str = dspy.InputField()
    setup: str = dspy.OutputField()
    punchline: str =dspy.OutputField()
    contradiction: str = dspy.OutputField()
    delivery: str = dspy.OutputField(desc="The full joke delivery should be in the comedian's voice")

In [11]:
joke_generator =dspy.Predict(JokeSignature)
joke=joke_generator(quest = "Write a joke about a conversation between a human and a cat.")
print(joke)

Prediction(
    setup='A human sits down for a serious conversation with their cat, trying to discuss boundaries, schedules, and who’s really in charge of the apartment.',
    punchline='The cat listened patiently, then knocked a glass off the table and meowed, “Counterproposal.”',
    contradiction='The human thought they were having a mature discussion; the cat treated it like a negotiation with no written rules and a lot of casualties.',
    delivery='So I tried to have a grown-up conversation with my cat. I said, “Listen, we need to talk about your behavior. The climbing, the yowling at 3 a.m., the way you stare at me like I’m the help.”\n\nHe just sat there, real calm, tail wrapped around himself like he was in a business meeting. I thought, wow, finally, respect.\n\nI said, “From now on, you sleep at night. I sleep at night. We both act like civilized roommates.”\n\nHe blinked once, walked to the edge of the table, knocked my glass onto the floor, and looked me dead in the eye li

In [13]:
print(kode_model.inspect_history(1))





[2026-09-23T22:33:23.795513]

System message:

Your input fields are:
1. `quest` (str):
Your output fields are:
1. `setup` (str): 
2. `punchline` (str): 
3. `contradiction` (str): 
4. `delivery` (str): The full joke delivery should be in the comedian's voice
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## quest ## ]]
{quest}

[[ ## setup ## ]]
{setup}

[[ ## punchline ## ]]
{punchline}

[[ ## contradiction ## ]]
{contradiction}

[[ ## delivery ## ]]
{delivery}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Yor are a comedian who likes to tell stories before delivering a punchline. You should sound funny.


User message:

[[ ## quest ## ]]
Write a joke about a conversation between a human and a cat.

Respond with the corresponding output fields, starting with the field `[[ ## setup ## ]]`, then `[[ ## punchline ## ]]`, then `[[ ## contradiction ## ]]`, then `[[ ## delivery ## ]]`, and then end

In [19]:
from pydantic import BaseModel, Field

In [20]:
class JokeIdea(BaseModel):
    setup: str
    contradiction: str
    punchline: str
    

In [21]:
class QueryToIdea(dspy.Signature):
    """You are a funny comedian and your goal is to generate a nice structure for a joke."""
    query: str = dspy.InputField()
    joke_idea: JokeIdea = dspy.OutputField()


In [22]:
class IdeaToJoke(dspy.Signature):
    """
    You are a funny comedian who likes to tell stories before delivering a punchline.
    You are always funny and act on the input joke idea.
    """
    joke_idea: JokeIdea = dspy.InputField()
    joke: str = dspy.OutputField(desc="The full joke delivery in the comedian's voice")

In [24]:
class JokeGenerator(dspy.Module):
    def __init__(self, n_attempts: int = 2):
        self.query_to_idea = dspy.Predict(QueryToIdea)
        self.idea_to_joke = dspy.Predict(IdeaToJoke)
        # self.n_attempts = n_attempts
    def forward(self, query: str):
        joke_idea = self.query_to_idea(query=query)
        print(f'Joke Idea:\n {joke_idea}')

        joke = self.idea_to_joke(joke_idea=joke_idea)
        print(f"Joke:\n{joke}")
        return joke

joke_generator = JokeGenerator()
joke= joke_generator(query= "Write a joke about a conversation between a human and a cat.")

print(joke.joke)


2026/09/23 22:54:02 WARNING dspy.predict.predict: Type mismatch for field 'joke_idea': expected JokeIdea based on given Signature, but the provided value is incompatible: Prediction(
    joke_idea=JokeIdea(setup='A human sits down with their cat and says, “We need to talk about your behavior.” The cat blinks slowly, as if to say, “Finally, someone is taking this relationship seriously.”', contradiction='The human thinks they’re the one leading the conversation, but the cat has already judged, interrupted, and emotionally outlasted them without saying a word.', punchline='After a long pause, the cat walks over to the food bowl, stares at it, then at the human, and the human says, “Oh, so now I’m the assistant in this meeting?”')
).


Joke Idea:
 Prediction(
    joke_idea=JokeIdea(setup='A human sits down with their cat and says, “We need to talk about your behavior.” The cat blinks slowly, as if to say, “Finally, someone is taking this relationship seriously.”', contradiction='The human thinks they’re the one leading the conversation, but the cat has already judged, interrupted, and emotionally outlasted them without saying a word.', punchline='After a long pause, the cat walks over to the food bowl, stares at it, then at the human, and the human says, “Oh, so now I’m the assistant in this meeting?”')
)
Joke:
Prediction(
    joke='So this human sits down with their cat like, “We need to talk about your behavior.”\n\nWhich is already funny, because if you’ve ever owned a cat, you know that’s not a conversation — that’s a performance review written by a tiny dictator in fur.\n\nThe cat just blinks slowly, like, “Finally. Someone is taking this relationship seriously.”\n\nAnd that’s the thing about cats: humans think 